# Distributed Training Basics

দুটি demo, দুটোই একটি একক CPU-তে চলে (কোনো সত্যিকারের multi-GPU cluster লাগে না -- অংশ 1-এর "device"গুলো কেবল একটি Python process-এর ভেতরের স্বাধীন model copy, যা আলাদা accelerator-এর প্রতিনিধি):

  1. স্ক্র্যাচ থেকে লেখা data-parallel training-এর একটি সিমুলেশন: একটি toy batch-কে `N`টি সিমুলেটেড device-এ ভাগ করুন, স্বাধীন model copy-তে প্রতি-device gradient গণনা করুন, সেগুলো গড় করুন (একটি all-reduce সিমুলেট করে), এবং যাচাই করুন ফলটি এক device-এ পুরো batch-এর ওপর সরাসরি গণনা করা gradient-এর সাথে মেলে।
  2. একটি memory-footprint calculator: সম্পূর্ণ replication বনাম প্রতিটি ZeRO sharding stage-এর তুলনা, বিভিন্ন model আকার ও device-সংখ্যার জন্য -- ZeRO paper-এর (Rajbhandari et al., 2020) প্রমিত 16*Psi-bytes-per-parameter হিসাব ব্যবহার করে।

**চালানোর নিয়ম:** কোষগুলো উপরে থেকে নিচে (Run All) চালান। প্রতিটি অংশের demo নিজের কোষেই চলে; শেষ কোষের `main()` পুরো রানটি একসাথে আরেকবার চালায়।

In [ ]:
import copy

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

## 1. Data-parallel gradient averaging, `N`টি "device"-এ সিমুলেটেড

একটি ছোট 2-layer MLP ("মডেল"-এর প্রতিনিধি), একটি toy batch, এবং `N`টি স্বাধীন model copy -- প্রতিটি নিজের batch shard-এ gradient গণনা করে। তারপর all-reduce (গড়) ফলটি পুরো batch-এর সরাসরি gradient-এর সাথে তুলনা করা হয়।

In [ ]:
# ---------------------------------------------------------------------------
# 1. Data-parallel gradient averaging, N সংখ্যক "device"-এ সিমুলেটেড
# ---------------------------------------------------------------------------

class TinyModel(nn.Module):
    """একটি ছোট 2-layer MLP -- এই সিমুলেশনে "the model"-এর প্রতিনিধি। যথেষ্ট
    ছোট যাতে full-batch ও sharded gradient হাতে হিসাব করা দ্রুত হয়, যথেষ্ট বড়
    যাতে তুলনা করার মতো একটি বাস্তব multi-parameter gradient থাকে।"""

    def __init__(self, d_in=8, d_hidden=16, d_out=1):
        super().__init__()
        self.fc1 = nn.Linear(d_in, d_hidden)
        self.fc2 = nn.Linear(d_hidden, d_out)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


def compute_gradient(model, x, y):
    """(x, y) তে MEAN-reduced MSE loss দিয়ে forward+backward চালায়, gradient-এর
    একটি flat vector রিটার্ন করে (একটি লম্বা vector, সব parameter সংযুক্ত)।"""
    model.zero_grad()
    pred = model(x)
    loss = F.mse_loss(pred, y)   # mean reduction -- এটিই shard-averaging-কে exact করে
    loss.backward()
    return torch.cat([p.grad.reshape(-1) for p in model.parameters()]), loss.item()


def data_parallel_demo():
    print("=" * 74)
    print("1. DATA-PARALLEL GRADIENT AVERAGING, SIMULATED ACROSS N DEVICES")
    print("=" * 74)

    batch_size, num_devices = 32, 4
    d_in = 8
    x = torch.randn(batch_size, d_in)
    y = torch.randn(batch_size, 1)

    base_model = TinyModel(d_in)

    # --- "একটি বিশাল device": gradient সরাসরি FULL batch-এর উপর গণনা করা ---
    single_device_model = copy.deepcopy(base_model)
    full_batch_grad, full_batch_loss = compute_gradient(single_device_model, x, y)

    # --- N সংখ্যক সিমুলেটেড device: প্রতিটিতে একটি IDENTICAL model copy (data
    # parallelism-এ "replicate the model" বলতে এটিই বোঝায়) এবং batch-এর একটি
    # DIFFERENT shard। প্রতিটি সম্পূর্ণ স্বাধীনভাবে নিজের local gradient গণনা করে। ---
    shard_size = batch_size // num_devices
    per_device_grads = []
    per_device_losses = []
    for device_id in range(num_devices):
        device_model = copy.deepcopy(base_model)   # প্রতিটি device-এ একই প্রারম্ভিক weight
        start = device_id * shard_size
        x_shard = x[start:start + shard_size]
        y_shard = y[start:start + shard_size]
        grad, loss_val = compute_gradient(device_model, x_shard, y_shard)
        per_device_grads.append(grad)
        per_device_losses.append(loss_val)
        print(f"  device {device_id}: batch shard = examples [{start}:{start + shard_size}), "
              f"local loss = {loss_val:.4f}")

    # --- All-reduce: N সংখ্যক local gradient-এর গড় -- data parallelism-কে প্রতি
    # training step-এ এই একটি communication ধাপই দরকার। ---
    all_reduced_grad = torch.stack(per_device_grads).mean(dim=0)

    max_abs_diff = (all_reduced_grad - full_batch_grad).abs().max().item()
    grad_norm = full_batch_grad.norm().item()

    print(f"\nFull-batch gradient norm (one big 'device'):         {grad_norm:.6f}")
    print(f"All-reduced (averaged) gradient norm ({num_devices} devices):  "
          f"{all_reduced_grad.norm().item():.6f}")
    print(f"Max absolute difference between the two gradient vectors: {max_abs_diff:.2e}")
    print(f"\n-> The difference is at floating-point-arithmetic noise level ({max_abs_diff:.1e}),")
    print(f"   not a real discrepancy: because the loss uses MEAN reduction and all")
    print(f"   {num_devices} shards are equal size, averaging {num_devices} independently-computed")
    print(f"   shard gradients is mathematically IDENTICAL to computing the gradient on")
    print(f"   the whole batch at once. This is exactly why data parallelism produces the")
    print(f"   same training trajectory as single-device training on the full batch --")
    print(f"   it just gets there by running the {num_devices} shards' forward/backward passes")
    print(f"   in PARALLEL instead of sequentially, then synchronizing with one all-reduce.")


# এই অংশের demo।
data_parallel_demo()

## 2. Memory-footprint calculator: সম্পূর্ণ replication বনাম ZeRO sharding stage

ZeRO paper-এর (Rajbhandari et al., 2020) প্রমিত mixed-precision + Adam byte-হিসাব ব্যবহার করে, `N` বাড়ার সঙ্গে প্রতিটি stage-এ per-device মেমরি কীভাবে কমে তা দেখানো হয়।

In [ ]:
# ---------------------------------------------------------------------------
# 2. Memory-footprint calculator: সম্পূর্ণ replication বনাম ZeRO sharding stages
# ---------------------------------------------------------------------------

# প্রমিত mixed-precision + Adam byte-হিসাব (Rajbhandari et al., 2020,
# "ZeRO", Section 3): প্রতি parameter-এ, একটি data-parallel replica-কে ধরে রাখতে হয়
#   fp16 parameters:      2 bytes
#   fp16 gradients:       2 bytes
#   fp32 master params:   4 bytes  |
#   fp32 Adam momentum:   4 bytes  |-- optimizer state, মোট 12 bytes
#   fp32 Adam variance:   4 bytes  |
# সম্পূর্ণ replication-এ মোট 16 bytes/parameter (ZeRO-এর নিজস্ব "16*Psi" হিসাব)।
BYTES_PARAMS_FP16 = 2
BYTES_GRADS_FP16 = 2
BYTES_OPTIMIZER_STATES = 12   # fp32 master weight + momentum + variance


def per_device_bytes(num_params, num_devices, stage):
    """stage: 'baseline' (সম্পূর্ণ replication), 1, 2, বা 3 (ZeRO stage)।"""
    if stage == "baseline":
        return num_params * (BYTES_PARAMS_FP16 + BYTES_GRADS_FP16 + BYTES_OPTIMIZER_STATES)
    if stage == 1:      # শুধু optimizer states shard করা
        return num_params * (BYTES_PARAMS_FP16 + BYTES_GRADS_FP16 + BYTES_OPTIMIZER_STATES / num_devices)
    if stage == 2:      # optimizer states + gradients shard করা
        return num_params * (BYTES_PARAMS_FP16 + (BYTES_GRADS_FP16 + BYTES_OPTIMIZER_STATES) / num_devices)
    if stage == 3:      # সবকিছু shard: params, gradients, optimizer states
        return num_params * (BYTES_PARAMS_FP16 + BYTES_GRADS_FP16 + BYTES_OPTIMIZER_STATES) / num_devices
    raise ValueError(stage)


def memory_calculator_demo():
    print("\n" + "=" * 74)
    print("2. PER-DEVICE MEMORY: FULL REPLICATION vs. ZeRO SHARDING STAGES")
    print("=" * 74)

    num_params = 7e9   # একটি 7B-parameter model -- LLaMA-7B / Mistral-7B স্কেল
    print(f"Model size: {num_params:.0e} parameters, mixed-precision + AdamW "
          f"(16 bytes/param fully replicated)\n")

    device_counts = [1, 2, 4, 8, 16, 32, 64]
    print(f"{'devices':>8}{'baseline (GB)':>16}{'ZeRO-1 (GB)':>14}"
          f"{'ZeRO-2 (GB)':>14}{'ZeRO-3 (GB)':>14}")
    for n in device_counts:
        baseline_gb = per_device_bytes(num_params, n, "baseline") / 1e9
        zero1_gb = per_device_bytes(num_params, n, 1) / 1e9
        zero2_gb = per_device_bytes(num_params, n, 2) / 1e9
        zero3_gb = per_device_bytes(num_params, n, 3) / 1e9
        print(f"{n:>8}{baseline_gb:>16.1f}{zero1_gb:>14.1f}{zero2_gb:>14.1f}{zero3_gb:>14.1f}")

    baseline_1 = per_device_bytes(num_params, 1, "baseline") / 1e9
    zero3_64 = per_device_bytes(num_params, 64, 3) / 1e9
    print(f"\n-> Full replication ('baseline') costs {baseline_1:.1f} GB per device NO MATTER")
    print(f"   how many devices you add -- more devices only help wall-clock time, not")
    print(f"   memory, under plain data parallelism. ZeRO-3, by contrast, drives per-device")
    print(f"   memory down to {zero3_64:.2f} GB at 64 devices -- roughly a {baseline_1 / zero3_64:.0f}x")
    print(f"   reduction -- because every device now stores only its 1/64 SHARD of the")
    print(f"   parameters, gradients, and optimizer states, reconstructing the full values")
    print(f"   on the fly only when a given layer actually needs them for compute.")
    print(f"   Notice ZeRO-1 and ZeRO-2 plateau far above ZeRO-3's curve: they still fully")
    print(f"   replicate parameters (and, for ZeRO-1, gradients too), so the un-sharded")
    print(f"   piece puts a floor under how low per-device memory can go.")


def main():
    data_parallel_demo()
    memory_calculator_demo()


# এই অংশের demo।
memory_calculator_demo()

In [ ]:
main()